# 🎸 Notebook 06: User Study — Evaluation Materials & Analysis

**Thesis:** A Conversational AI System for Symbolic Guitar Strumming Pattern and Chord Progression Generation from Natural Language Prompts

**Author:** Rohan Rajendra Dhanawade | SRH Berlin University of Applied Sciences

**Chat:** 11 (User Study Materials)

---

## What This Notebook Does

This notebook runs the complete user study pipeline:

1. **Generate 20 evaluation samples** from your hybrid system
2. **Create study materials** (PDF chord sheets, questionnaire, consent form)
3. **Create rating templates** for you + participants
4. **Analyze completed ratings** and produce thesis-ready tables & charts

### Evaluation Approach

We use a **combined evaluation methodology**:
- **Developer heuristic evaluation** (you rate all 20 samples using a structured rubric)
- **Pilot participant study** (2-3 guitarist friends also rate the samples)
- **11 objective metrics** (already computed in Chat 10)

This is framed in the thesis as a "preliminary evaluation" — standard practice for master's theses when full user studies aren't feasible within the timeline.

## Step 0: Environment Setup

First, clone the repo and install dependencies.

In [ ]:
# Clone repository (uncomment if running fresh)
# !git clone -b development https://github.com/rohand575/guitar-strum-gen.git
# %cd guitar-strum-gen

# If already cloned, just navigate
import os
if os.path.exists('guitar-strum-gen'):
    os.chdir('guitar-strum-gen')
    print(f'Working directory: {os.getcwd()}')
elif os.path.exists('src'):
    print(f'Already in project root: {os.getcwd()}')
else:
    print('Please clone the repo first (uncomment the git clone line above)')

In [ ]:
# Install dependencies
!pip install torch transformers pydantic reportlab matplotlib numpy -q

# Verify imports
import sys
sys.path.insert(0, '.')

from src.data.schema import GuitarSample
print('Schema loaded')

from src.rules.generate_rule_based import generate_rule_based
print('Rule-based generator loaded')

# Test quick generation
test = generate_rule_based('upbeat rock in G major', verbose=False)
print(f'Test generation: {test.chords} | {test.strum_pattern}')
print('\nAll imports successful!')

---
## Step 1: Generate 20 Evaluation Samples

We run 20 diverse prompts through the system. These cover all 9 genres and 8 emotions.

**Choose your generation mode:**
- `hybrid` — Uses the full neural + rule-based system (requires trained checkpoint)
- `rule_based` — Uses only the rule-based system (always works, less creative)
- `from_test` — Maps prompts to existing test.jsonl data (no generation needed)

In [ ]:
from src.evaluation.sample_generator import (
    run_sample_generation, EVALUATION_PROMPTS, print_coverage_report
)

# ============================================================
# CHOOSE YOUR MODE HERE
# ============================================================
# Option 1: Full hybrid system (recommended if checkpoint exists)
GENERATION_MODE = 'hybrid'
CHECKPOINT_PATH = 'checkpoints/guitar_lstm_final.pt'

# Option 2: Rule-based only (if no checkpoint)
# GENERATION_MODE = 'rule_based'

# Option 3: Use existing test data (no generation needed)
# GENERATION_MODE = 'from_test'
# ============================================================

# Check if checkpoint exists for hybrid mode
if GENERATION_MODE == 'hybrid' and not os.path.exists(CHECKPOINT_PATH):
    print(f'Checkpoint not found at {CHECKPOINT_PATH}')
    print('Falling back to rule_based mode...')
    GENERATION_MODE = 'rule_based'

print(f'Generation mode: {GENERATION_MODE}')
print(f'Number of prompts: {len(EVALUATION_PROMPTS)}\n')

In [ ]:
# Generate all 20 samples
samples = run_sample_generation(
    mode=GENERATION_MODE,
    checkpoint_path=CHECKPOINT_PATH if GENERATION_MODE == 'hybrid' else '',
    test_path='data/processed/test.jsonl',
    output_dir='evaluation_samples',
    verbose=True
)

print(f'\nGenerated {len(samples)} samples successfully!')

### Preview: First 3 Samples

Let's verify the outputs look correct before generating study materials.

In [ ]:
import json

for i, sample in enumerate(samples[:3]):
    if 'error' in sample:
        print(f'Sample {i+1}: ERROR - {sample["error"]}')
        continue
    
    chords = ' - '.join(sample.get('generated_chords', []))
    pattern = sample.get('generated_strum_pattern', '')
    genre = sample.get('generated_genre', '?')
    emotion = sample.get('generated_emotion', '?')
    source = sample.get('generation_source', '?')
    
    print(f'Sample {i+1} [{genre}/{emotion}] (source: {source})')
    print(f'  Prompt:  "{sample["prompt"][:60]}..."')
    print(f'  Chords:  {chords}')
    print(f'  Strum:   {pattern}')
    print(f'  Key:     {sample.get("generated_key", "?")} {sample.get("generated_mode", "?")}')
    print(f'  Tempo:   {sample.get("generated_tempo", "?")} BPM')
    print()

---
## Step 2: Generate Study Materials

This creates all the documents needed for the evaluation:

| File | Description |
|------|-------------|
| `chord_sheets.pdf` | Professional PDF with one chord sheet per sample (20 pages) |
| `evaluation_questionnaire.pdf` | Printable questionnaire with Likert scales |
| `consent_form.pdf` | Informed consent form for participants |
| `response_template.csv` | Digital version for Google Sheets |
| `all_chord_sheets.txt` | Text version of all chord sheets |
| `sample_01.txt` ... `sample_20.txt` | Individual text chord sheets |

In [ ]:
from src.evaluation.user_study_materials import generate_all_materials

materials = generate_all_materials(
    samples_path='evaluation_samples/samples.json',
    output_dir='evaluation_samples'
)

print('\nGenerated files:')
for name, path in materials.items():
    size = os.path.getsize(path)
    print(f'  {name:25s} -> {path} ({size:,} bytes)')

### Download Study Materials

Run the cell below to download the materials from Colab.
Send the **chord_sheets.pdf** + **response_template.csv** to your friends.

In [ ]:
# Download files from Colab
try:
    from google.colab import files
    
    print('Downloading study materials...\n')
    
    files.download('evaluation_samples/chord_sheets.pdf')
    files.download('evaluation_samples/evaluation_questionnaire.pdf')
    files.download('evaluation_samples/consent_form.pdf')
    files.download('evaluation_samples/response_template.csv')
    files.download('evaluation_samples/all_chord_sheets.txt')
    files.download('evaluation_samples/samples.json')
    
    print('\nAll files downloaded!')
    
except ImportError:
    print('Not running in Colab — files are saved in evaluation_samples/')
    print('You can find them there directly.')

---
## Step 3: Create Rating Templates

This creates JSON files where you (and your participants) fill in ratings.

Each template has:
- The complete evaluation rubric (what each 1-5 score means)
- All 20 samples with their outputs
- Empty fields (`null`) for you to replace with 1-5 ratings

### The Rubric (4 Dimensions)

| Dimension | What It Measures |
|-----------|------------------|
| **Playability** | Can a guitarist physically play this comfortably? |
| **Expressiveness** | Does it capture the mood described in the prompt? |
| **Usefulness** | Would you use this for practice or songwriting? |
| **Overall Quality** | General impression of musical quality |

In [ ]:
from src.evaluation.heuristic_evaluation import create_rating_template

# Create templates for each evaluator
# -----------------------------------
# 'developer' = you (the thesis author)
# 'participant_01', 'participant_02' = your friends
# Add more if you have more participants

evaluators = ['developer', 'participant_01', 'participant_02']

for eval_id in evaluators:
    create_rating_template(
        samples_path='evaluation_samples/samples.json',
        output_dir='evaluation_samples',
        evaluator_id=eval_id
    )
    print()

In [ ]:
# Download rating templates
try:
    from google.colab import files
    for eval_id in evaluators:
        files.download(f'evaluation_samples/ratings_{eval_id}.json')
    print('Templates downloaded!')
except ImportError:
    print('Templates saved in evaluation_samples/')

### How to Fill In Ratings

1. Open `ratings_developer.json` in any text editor
2. For each sample in the `"ratings"` array, replace `null` with a number 1-5:

```json
{
    "sample_num": 1,
    "prompt": "I need an energetic rock progression...",
    "chords": "A - D - E - A",
    "strum_pattern": "D_DU_DU_",
    "playability": 4,        <-- was null, now filled
    "expressiveness": 5,     <-- was null, now filled  
    "usefulness": 4,         <-- was null, now filled
    "overall_quality": 4,    <-- was null, now filled
    "comments": "Good energy, chords flow well"
}
```

3. Also fill in `guitar_experience_years` and `skill_level` at the top
4. Save the file
5. For friends: send them `chord_sheets.pdf` + their `ratings_participant_XX.json`

**Alternative:** Your friends can use `response_template.csv` in Google Sheets instead.

---
## Step 4: Upload Completed Ratings

After everyone has filled in their ratings, upload the JSON files here.

In [ ]:
# Upload completed rating files
try:
    from google.colab import files
    
    print('Upload your completed ratings_*.json files:')
    print('(You can select multiple files at once)\n')
    
    uploaded = files.upload()
    
    # Move uploaded files to the right directory
    for filename, content in uploaded.items():
        dest = os.path.join('evaluation_samples', filename)
        with open(dest, 'wb') as f:
            f.write(content)
        print(f'  Saved: {dest}')
    
    print(f'\nUploaded {len(uploaded)} rating file(s)')
    
except ImportError:
    print('Not in Colab — make sure your rating files are in evaluation_samples/')
    print('Expected files: ratings_developer.json, ratings_participant_01.json, etc.')

---
## Step 5: Analyze Results

This computes all statistics, generates tables, and creates thesis-ready visualizations.

In [ ]:
from src.evaluation.heuristic_evaluation import run_analysis

# Run the full analysis pipeline
stats = run_analysis(
    ratings_dir='evaluation_samples',
    output_dir='evaluation_samples'
)

### Visualizations for Thesis

In [ ]:
from IPython.display import Image, display
import matplotlib.pyplot as plt

chart_files = [
    ('Mean Ratings per Dimension', 'evaluation_samples/eval_dimension_ratings.png'),
    ('Ratings by Genre', 'evaluation_samples/eval_genre_ratings.png'),
    ('Evaluation Radar Profile', 'evaluation_samples/eval_radar.png'),
]

for title, path in chart_files:
    if os.path.exists(path):
        print(f'\n{title}:')
        display(Image(filename=path, width=600))
    else:
        print(f'  {path} not found (run analysis first)')

### LaTeX Tables (Copy-Paste into Thesis)

In [ ]:
# Display the LaTeX tables
latex_path = 'evaluation_samples/evaluation_tables.tex'

if os.path.exists(latex_path):
    with open(latex_path, 'r') as f:
        latex_content = f.read()
    print(latex_content)
else:
    print('LaTeX tables not generated yet. Run the analysis step first.')

### Detailed Results Breakdown

In [ ]:
import json

if stats:
    # Overall summary
    print('=' * 55)
    print('  THESIS-READY SUMMARY')
    print('=' * 55)
    print(f'  Evaluators:     {stats["n_evaluators"]}')
    print(f'  Samples rated:  {stats["n_samples_rated"]}')
    print(f'  Total ratings:  {stats["n_evaluators"] * stats["n_samples_rated"] * 4}')
    print()
    
    # Grand means
    print('  Grand Mean Ratings (1-5 Likert Scale):')
    print('  ' + '-' * 45)
    for dim in ['playability', 'expressiveness', 'usefulness', 'overall_quality']:
        if dim in stats.get('per_dimension', {}):
            d = stats['per_dimension'][dim]
            bar = '█' * int(d['mean'] * 4)  # Visual bar
            print(f'  {dim:<20s}: {d["mean"]:.2f} ± {d["std"]:.2f}  {bar}')
    print('  ' + '-' * 45)
    
    # Per-evaluator comparison
    if stats.get('per_evaluator'):
        print('\n  Per-Evaluator Means:')
        print('  ' + '-' * 55)
        print(f'  {"Evaluator":<20s} {"Play":>6s} {"Expr":>6s} {"Use":>6s} {"Overall":>8s}')
        print('  ' + '-' * 55)
        for ev in stats['per_evaluator']:
            m = ev['means']
            print(f'  {ev["evaluator_id"]:<20s} '
                  f'{m.get("playability", 0):>6.2f} '
                  f'{m.get("expressiveness", 0):>6.2f} '
                  f'{m.get("usefulness", 0):>6.2f} '
                  f'{m.get("overall_quality", 0):>8.2f}')
        print('  ' + '-' * 55)
else:
    print('No stats available. Complete the ratings and run analysis first.')

---
## Step 6: Download All Results

Download everything for your thesis.

In [ ]:
# Download all analysis outputs
try:
    from google.colab import files
    
    result_files = [
        'evaluation_samples/evaluation_analysis.json',
        'evaluation_samples/evaluation_tables.tex',
        'evaluation_samples/eval_dimension_ratings.png',
        'evaluation_samples/eval_genre_ratings.png',
        'evaluation_samples/eval_radar.png',
    ]
    
    for path in result_files:
        if os.path.exists(path):
            files.download(path)
            print(f'  Downloaded: {path}')
    
    print('\nAll results downloaded!')
    
except ImportError:
    print('Not in Colab — results are in evaluation_samples/')

---
## Thesis Writing Guidance

### How to Frame This in Your Methods Chapter

```
5.2 Subjective Evaluation

To complement the objective metrics (Section 5.1), a preliminary 
human evaluation was conducted. The evaluation instrument uses a 
5-point Likert scale (1 = Strongly Disagree to 5 = Strongly Agree) 
across four dimensions:

1. Playability: physical comfort and feasibility on guitar
2. Expressiveness: alignment with the mood/genre in the prompt  
3. Usefulness: practical value for practice or songwriting
4. Overall Quality: general musical quality impression

The evaluation followed a heuristic evaluation methodology 
(Nielsen, 1994), conducted by the developer (N=1) with domain 
expertise in both guitar performance and the system architecture. 
Additionally, a pilot study with N=2 intermediate-to-advanced 
guitarists provided external validation. Each evaluator independently 
rated 20 system-generated samples covering all 9 genres and 
8 emotion categories.

The complete study protocol, including consent forms, evaluation 
rubrics, and chord sheets, is provided in Appendix X.
```

### Key References to Cite

- **Nielsen, J. (1994).** Usability inspection methods. *Conference companion on Human factors in computing systems.* — For heuristic evaluation methodology
- **Likert, R. (1932).** A technique for the measurement of attitudes. *Archives of Psychology.* — For the Likert scale
- **Yang, L.-C., & Lerch, A. (2020).** On the evaluation of generative models in music. *Neural Computing and Applications.* — For music generation evaluation practices

### Limitations to Acknowledge

In the Discussion chapter, be transparent:

```
The subjective evaluation has several limitations. The small 
sample size (N=3) limits statistical generalizability. The 
developer evaluation, while structured through a formal rubric, 
may carry inherent bias toward the system. Future work should 
include a larger-scale study with diverse guitarist populations 
and blind evaluation conditions.
```

---
## Summary: Chat 11 Deliverables

### Files Created

| File | Location | Purpose |
|------|----------|---------|
| `sample_generator.py` | `src/evaluation/` | 20 curated prompts + 3 generation modes |
| `user_study_materials.py` | `src/evaluation/` | PDF/text chord sheets, questionnaire, consent form |
| `heuristic_evaluation.py` | `src/evaluation/` | Rating rubric, analysis, LaTeX tables, charts |
| `06_user_study.ipynb` | `notebooks/` | This notebook — complete pipeline |

### Generated Outputs

| Output | Description |
|--------|-------------|
| `chord_sheets.pdf` | 20-page professional chord sheet document |
| `evaluation_questionnaire.pdf` | Printable rating questionnaire |
| `consent_form.pdf` | Informed consent for participants |
| `response_template.csv` | Google Sheets-compatible response form |
| `ratings_*.json` | Rating templates for each evaluator |
| `evaluation_analysis.json` | Complete statistical results |
| `evaluation_tables.tex` | LaTeX tables for thesis |
| `eval_*.png` | Thesis-ready visualization charts |

### Next Step → Chat 12: Thesis Writing Support

With all code, evaluation, and materials complete, Chat 12 will help compile everything into the formal thesis document.